In [60]:
# Paramètres
import io
import pandas as pd
import boto3
import pandas as pd
import io
import pandas as pd
import unicodedata
import re
import numpy as np
import unicodedata, re
from pathlib import Path
from pandas.api.types import is_string_dtype
from collections import Counter
from rapidfuzz import process, fuzz
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

In [2]:
# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://192.168.1.230:30137", # service interne K8s
 aws_access_key_id="datalab-team",
 aws_secret_access_key="minio-datalabteam123",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)
# Paramètres
bucket_name = "gold"
object_key = "panel_admin/panel_complet.parquet"
# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
# Afficher un aperçu
panel_solde_df.head()

,cle_unique,matricule,nom,sexe,date_naissance,situation_matrimoniale,nombre_enfant,situation_normalisee,CODE_ORGANISME,organisme,...,annee,mois,montant_brut,montant_net,retenue_pension,impot,charge_patronale,montant_brut_SOURCE,montant_net_SOURCE,fichier_source
0,011242X15,011242X,DOSSO MEGBO,Masculin,1939-01-01,Marié,0,en_activite,15,Min. d'Etat Administration du Territoire,...,2015,1,34560.0,34145.0,0.0,415.0,0.0,OBSERVE,OBSERVE,012015.xlsx
1,011242X15,011242X,DOSSO MEGBO,Masculin,1939-01-01,Marié,0,en_activite,15,Min. d'Etat Administration du Territoire,...,2015,2,34560.0,34146.0,0.0,414.0,0.0,OBSERVE,OBSERVE,022015.xlsx
2,011242X15,011242X,DOSSO MEGBO,Masculin,1939-01-01,Marié,0,en_activite,15,Min. d'Etat Administration du Territoire,...,2015,3,34560.0,34145.0,0.0,415.0,0.0,OBSERVE,OBSERVE,032015.xlsx
3,011242X15,011242X,DOSSO MEGBO,Masculin,1939-01-01,Marié,0,en_activite,15,Min. d'Etat Administration du Territoire,...,2015,4,34560.0,34145.0,0.0,415.0,0.0,OBSERVE,OBSERVE,042015.xlsx
4,011242X15,011242X,DOSSO MEGBO,Masculin,1939-01-01,Marié,0,en_activite,15,Min. d'Etat Administration du Territoire,...,2015,5,34560.0,34145.0,0.0,415.0,0.0,OBSERVE,OBSERVE,052015.xlsx


In [18]:
print("Number of variables (columns) in the dataset:", panel_solde_df.shape[1])
print("Number of observations (rows) in the dataset:", panel_solde_df.shape[0])
print("Column names (variables) in the dataset:", panel_solde_df.columns.tolist())

Number of variables (columns) in the dataset: 41
Number of observations (rows) in the dataset: 32503905
Column names (variables) in the dataset: ['cle_unique', 'matricule', 'nom', 'sexe', 'date_naissance', 'situation_matrimoniale', 'nombre_enfant', 'situation_normalisee', 'CODE_ORGANISME', 'organisme', 'CODE_EMPLOI', 'emploi', 'Code_CITP', 'Metier_CITP', 'CODE_GRADE', 'grade', 'statut_fonctionnaire', 'CODE_FONCTION', 'fonction', 'CODE_SERVICE', 'service', 'CODE_AFFECTATION', 'lieu_affectation', 'CODE_POSTE', 'prise_service', 'date_retraite', 'age_retraite', 'GRADE', 'GRADE_SOURCE', 'mois_annee', 'annee', 'mois', 'montant_brut', 'montant_net', 'retenue_pension', 'impot', 'charge_patronale', 'montant_brut_SOURCE', 'montant_net_SOURCE', 'fichier_source', 'fichier_source_check']


In [55]:
# Check matricule uniqueness by mois_annee
counts = panel_solde_df.groupby('mois_annee').agg(
    total_rows=('matricule', 'size'),
    unique_matricules=('matricule', 'nunique')
)
counts['has_duplicates'] = counts['total_rows'] > counts['unique_matricules']
if counts['has_duplicates'].sum() == 0:
    print("Unique")
else:
    print("Not unique")

Unique


In [40]:
tab_code_grade = panel_solde_df['CODE_GRADE'].value_counts(dropna=False).reset_index()
tab_code_grade.columns = ['CODE_GRADE', 'count']

tab_grade_source = panel_solde_df['GRADE_SOURCE'].value_counts(dropna=False).reset_index()
tab_grade_source.columns = ['GRADE_SOURCE', 'count']

tab_grade = panel_solde_df['GRADE'].value_counts(dropna=False).reset_index()
tab_grade.columns = ['GRADE', 'count']

with pd.ExcelWriter(r'C:\Users\a_coulibaly\OneDrive - Metrics & Decisions\INS\CAE\Banque de données travailleurs\Data\Solde\Output\tabulations_Grade.xlsx', engine='openpyxl') as writer:
	tab_code_grade.to_excel(writer, sheet_name='CODE_GRADE', index=False)
	tab_grade_source.to_excel(writer, sheet_name='GRADE_SOURCE', index=False)
	tab_grade.to_excel(writer, sheet_name='GRADE', index=False)

In [ ]:
# Tabulation de emploi et CODE_EMPLOI
tab_emploi = panel_solde_df["emploi"].value_counts(dropna=False).reset_index()
tab_emploi.columns = ["emploi", "count"]

tab_code_emploi = panel_solde_df["CODE_EMPLOI"].value_counts(dropna=False).reset_index()
tab_code_emploi.columns = ["CODE_EMPLOI", "count"]

tab_emploi.head(10), tab_code_emploi.head(10)

(                                  emploi    count
 0            instituteur décret 2009-208  6996443
 1    Instituteur Adjoint décret 2009-208  3324533
 2                Sous-0fficier de Police  2254407
 3               Infirmier Diplomé d'Etat  1350555
 4    Professeur de Lycée décret 2009-208  1281939
 5  Professeur de Collège décret 2009-208  1125112
 6             Sage-Femme Diplomée d'État   793139
 7        Professeur de Lycée barême 100%   771215
 8      Professeur de Collège barême 100%   749135
 9       Inspecteur Principal: Professeur   555945,
   CODE_EMPLOI    count
 0        28YF  6996443
 1        29YT  3324533
 2        P3ZI  2254407
 3        08IS  1350555
 4        25EG  1281939
 5        27EG  1125112
 6        73JK   793139
 7        15EG   771215
 8        17EG   749135
 9        24EG   555945)

In [57]:
tab_annee = panel_solde_df['annee'].value_counts(dropna=False).reset_index()
tab_annee.columns = ['annee', 'count']
tab_annee

,annee,count
0,2025,3977686
1,2024,3719583
2,2023,3461563
3,2022,3196850
4,2021,2978710
5,2020,2822419
6,2019,2662326
7,2018,2578098
8,2017,2460161
9,2016,2382162


In [56]:
# Calculate statistics for gross and net amounts per sex for 2024 and 2025
avg_by_sexe_2024_2025 = panel_solde_df[panel_solde_df['annee'].isin([2024, 2025])].groupby(['sexe'])[['montant_brut', 'montant_net']].agg(['mean', 'median', 'min', 'max']).reset_index()
avg_by_sexe_2024_2025.columns = ['sexe'] + [f'{col[0]}_{col[1]}' for col in avg_by_sexe_2024_2025.columns[1:]]  
avg_by_sexe_2024_2025      

,sexe,montant_brut_mean,montant_brut_median,montant_brut_min,montant_brut_max,montant_net_mean,montant_net_median,montant_net_min,montant_net_max
0,Féminin,439479.380659,377404.0,1516.0,47387233.0,350278.603929,298799.0,0.0,39532694.0
1,Masculin,509880.884194,429275.0,156.0,62781031.0,407119.884389,346647.0,0.0,52761700.0


In [58]:
# Calculate statistics for gross and net amounts per sex for 2024 and 2025 for fonctionnaires (GRADE_SOURCE != 'NF_NON_IMPUTABLE')
avg_by_sexe_2024_2025 = panel_solde_df[(panel_solde_df['annee'].isin([2024, 2025])) & (panel_solde_df['GRADE_SOURCE'] != 'NF_NON_IMPUTABLE')].groupby(['sexe'])[['montant_brut', 'montant_net']].agg(['mean', 'median', 'min', 'max']).reset_index()
avg_by_sexe_2024_2025.columns = ['sexe'] + [f'{col[0]}_{col[1]}' for col in avg_by_sexe_2024_2025.columns[1:]]
avg_by_sexe_2024_2025

,sexe,montant_brut_mean,montant_brut_median,montant_brut_min,montant_brut_max,montant_net_mean,montant_net_median,montant_net_min,montant_net_max
0,Féminin,441637.065128,379719.0,1516.0,47387233.0,351687.953314,299429.0,0.0,39532694.0
1,Masculin,513272.204832,430052.0,156.0,62781031.0,409317.384548,347986.0,0.0,52761700.0


In [ ]:
# Calculate statistics for gross and net amounts per sex and grade
avg_by_sexe_grade = panel_solde_df.groupby(['sexe', 'GRADE'])[['montant_brut', 'montant_net']].agg(['mean', 'median', 'min', 'max']).reset_index()
avg_by_sexe_grade.columns = ['sexe', 'GRADE'] + [f'{col[0]}_{col[1]}' for col in avg_by_sexe_grade.columns[2:]]
# Calculate statistics for gross and net amounts per grade
avg_by_grade = panel_solde_df.groupby(['GRADE'])[['montant_brut', 'montant_net']].agg(['mean', 'median', 'min', 'max']).reset_index()
avg_by_grade.columns = ['GRADE'] + [f'{col[0]}_{col[1]}' for col in avg_by_grade.columns[1:]]
avg_by_grade

In [47]:
tab_statut_fonctionnaire = panel_solde_df['statut_fonctionnaire'].value_counts(dropna=False).reset_index()
tab_statut_fonctionnaire.columns = ['statut_fonctionnaire', 'count']
tab_statut_fonctionnaire.head(10)

,statut_fonctionnaire,count
0,Fonctionnaire Catégorie B3,9271623
1,Fonctionnaire Catégorie C3,4028300
2,Fonctionnaire Catégorie A3,3564450
3,Fonctionnaire Catégorie A7,3407507
4,Fonctionnaire Catégorie C Titulaire,3343861
5,Fonctionnaire Catégorie B Titulaire,2969626
6,Fonctionnaire Catégorie A4,2104386
7,Fonctionnaire Catégorie A5,935927
8,Fonctionnaire Catégorie C1,873814
9,Fonctionnaire Catégorie D1,436993


In [37]:
# Load the correspondance grille emploi Excel file from S3
object_key_corresp = "panel_admin/references/CORRESPONDANCE_GRILLE_EMPLOI.xlsx"
obj_corresp = s3_client.get_object(Bucket=bucket_name, Key=object_key_corresp)
df_corresp = pd.read_excel(io.BytesIO(obj_corresp['Body'].read()))
df_corresp.head(10)

,EMPLOI,GRADE,BAREME / GRILLE SALARIALE
0,Inspecteur de la Jeunesse et des Sports décret...,A4,ENS SEC & PRIM
1,DIRECTEUR DE RECHERCHE,A7,ENS SUP
2,Tech. Sup. de Chimie,B3,Barême Général
3,KINESITHERAPEUTE,B3,Barême Général
4,INGENIEUR PRINCIPAL: AGRONOMIE,A5,Barême Général
5,INGENIEUR PRINCIPAL: CONSTRUCTION,A5,Barême Général
6,INGENIEUR GENERAL DES SERVICES DE SANTE,A7,Barême Général
7,Agent Spécialisé des Travaux publics,D1,Barême Général
8,Agent d'Imprimerie,C1,Barême Général
9,Agent de Maîtrise des Travaux Publics,C2,Barême Général


In [38]:
panel_solde_df = pd.merge(panel_solde_df, df_corresp, left_on=['emploi', 'GRADE'], right_on=['EMPLOI', 'GRADE'], how='left')
# Check for unmatched records after the join
print("Nombre de lignes après fusion:", len(panel_solde_df))
print("Nombre de colonnes après fusion:", panel_solde_df.shape[1])
print("Colonnes avec NaN après fusion:")
print(panel_solde_df.isnull().sum())  

Nombre de lignes après fusion: 32503905
Nombre de colonnes après fusion: 42
Colonnes avec NaN après fusion:
cle_unique                          0
matricule                           0
nom                                 0
sexe                              748
date_naissance                   6106
situation_matrimoniale              0
nombre_enfant                   26876
situation_normalisee                0
CODE_ORGANISME                  11019
organisme                           0
CODE_EMPLOI                    532996
emploi                         508757
Code_CITP                      546793
Metier_CITP                    546793
CODE_GRADE                     511964
grade                          508595
statut_fonctionnaire              247
CODE_FONCTION                27178510
fonction                            0
CODE_SERVICE                   369605
service                            17
CODE_AFFECTATION                92905
lieu_affectation                    0
CODE_POSTE        

In [ ]:

# Perform the join between panel_solde_df and df_corresp on 'emploi'
panel_solde_df = pd.merge(panel_solde_df, df_corresp, on='emploi', how='left')

In [7]:
# Fréquence des valeurs dans la colonne montant_brut_SOURCE
tab = panel_solde_df["montant_brut_SOURCE"].value_counts(dropna=False).reset_index()
tab.columns = ["montant_brut_SOURCE", "count"]
tab.head(100)

,montant_brut_SOURCE,count
0,OBSERVE,32503905


In [19]:
# Fréquence des valeurs dans la colonne montant_brut_SOURCE
tab = panel_solde_df["montant_net_SOURCE"].value_counts(dropna=False).reset_index()
tab.columns = ["montant_net_SOURCE", "count"]
tab.head(100)

,montant_net_SOURCE,count
0,OBSERVE,32503905


In [13]:
# vérifier l'unicité de `matricule` par `mois_annee`
counts = panel_solde_df.groupby('mois_annee').agg(
	total_rows=('matricule', 'size'),
	unique_matricules=('matricule', 'nunique')
)
counts['has_duplicates'] = counts['total_rows'] > counts['unique_matricules']
counts['duplicated_items'] = counts['total_rows'] - counts['unique_matricules']
print(counts.head(20))
print("\nNombre de mois_annee avec doublons matricule :", counts['has_duplicates'].sum())


            total_rows  unique_matricules  has_duplicates  duplicated_items
mois_annee                                                                 
012015          183482             183482           False                 0
012016          196670             196670           False                 0
012017          200940             200940           False                 0
012018          210010             210010           False                 0
012019          215750             215750           False                 0
012020          226228             226228           False                 0
012021          245380             245380           False                 0
012022          258900             258900           False                 0
012023          279921             279921           False                 0
012024          295960             295960           False                 0
012025          318880             318880           False                 0
022015      

In [16]:
# Check if fichier_source is mois_annee followed by '.xlsx'
panel_solde_df['fichier_source'] = panel_solde_df['mois_annee']
is_same = (panel_solde_df['fichier_source'] == panel_solde_df['fichier_source']).all()
print("fichier_source always equal to mois_annee", is_same)

fichier_source always equal to mois_annee True


In [21]:
# Nombre d'observations par période (mois_annee)
observations_par_periode = panel_solde_df.groupby('mois_annee').size().reset_index(name='count')
observations_par_periode
observations_par_periode.to_excel('observations_par_periode.xlsx', index=False)

In [28]:
# Paramètres
bucket_name = "staging"
object_key = "panel_admin/references/anstat_2025.txt.xlsx"
# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
# Charger dans un DataFrame depuis le fichier Excel
df = pd.read_excel(io.BytesIO(obj['Body'].read()))
# Afficher un aperçu
df.head()

,SECTEUR,CATEGORIE,GRADE,INDICE,ECHELON/CLASSE,Salaire de Base,Résidence,Logement,Transport,Allocations\n (2 enfants),AUTRES INDEMNITES,Montant brut,Charge patronale,ITS,PENSION,MUGEF-CI/CMU,NET,BASE IMPOSABLE
0,Barême Général,A,A7,2350,21,548621.60,82293.240,90000,15000,15000,NaN,750914.840,91455.220720,91992.11640,45700.179280,7004.0,606218.544320,630914.840
1,Barême Général,A,A7,2295,33,535781.52,80367.228,90000,15000,15000,NaN,736148.748,89314.779384,88891.23708,44630.600616,7004.0,595622.910304,616148.748
2,Barême Général,A,A7,2255,32,526443.28,78966.492,90000,15000,15000,NaN,725409.772,87758.094776,86636.05212,43852.725224,7004.0,587916.994656,605409.772
3,Barême Général,A,A7,2170,31,506599.52,75989.928,90000,15000,15000,NaN,702589.448,84450.139984,81843.78408,42199.740016,7004.0,571541.923904,582589.448
4,Barême Général,A,A7,2135,44,498428.56,74764.284,90000,15000,15000,NaN,693192.844,83088.040952,79870.49724,41519.099048,7004.0,564799.247712,573192.844


In [29]:
df.columns

Index(['SECTEUR', 'CATEGORIE', 'GRADE', 'INDICE', 'ECHELON/CLASSE',
       'Salaire de Base', 'Résidence', 'Logement', 'Transport',
       'Allocations\n (2 enfants)', 'AUTRES INDEMNITES', 'Montant brut',
       'Charge patronale', 'ITS', 'PENSION', 'MUGEF-CI/CMU', 'NET',
       'BASE IMPOSABLE'],
      dtype='object')

In [30]:
# Calculate statistics for gross and net amounts per sector and grade
avg_by_secteur_grade = df.groupby(['SECTEUR', 'GRADE'])[['Montant brut', 'NET']].agg(['mean', 'median', 'min', 'max']).reset_index()
avg_by_secteur_grade.columns = ['SECTEUR', 'GRADE'] + [f'{col[0]}_{col[1]}' for col in avg_by_secteur_grade.columns[2:]]
avg_by_secteur_grade

,SECTEUR,GRADE,Montant brut_mean,Montant brut_median,Montant brut_min,Montant brut_max,NET_mean,NET_median,NET_min,NET_max
0,Barême Général,A3,434989.604000,426935.372,312833.752,570569.176,371170.698469,365269.103456,283726.382896,468335.725248
1,Barême Général,A4,498891.002769,486603.136,340284.588,662453.868,419815.275825,410908.963328,306651.569024,537093.332064
2,Barême Général,A5,534379.910462,528081.088,379077.796,687823.356,450859.491210,446320.794624,339646.663808,560946.289888
3,Barême Général,A6,589107.384308,593857.316,473043.836,695877.588,490111.142201,493519.527968,406827.976928,566725.726624
4,Barême Général,A7,699569.111000,697891.146,646209.824,750914.840,569374.635128,568170.585808,531085.866752,606218.544320
...,...,...,...,...,...,...,...,...,...,...
60,para militaire,A7,919569.111000,917891.146,866209.824,970914.840,789374.635128,788170.585808,751085.866752,826218.544320
61,para militaire,B3,522088.478462,518474.400,432562.592,605728.580,478781.034428,476144.211200,413866.193216,538754.455840
62,para militaire,C1,367247.882462,366318.548,343498.224,391823.616,352740.764998,351902.990184,331330.964192,374895.254528
63,para militaire,C2,377470.561538,375715.152,347525.340,406589.708,361792.438403,360373.824416,334961.321720,386852.212184


In [32]:
# Calculate statistics for gross and net amounts per grade
avg_by_grade = df.groupby('GRADE')[['Montant brut', 'NET']].agg(['mean', 'median', 'min', 'max']).reset_index()
avg_by_grade.columns = ['GRADE'] + [f'{col[0]}_{col[1]}' for col in avg_by_grade.columns[1:]]
avg_by_grade

,GRADE,Montant brut_mean,Montant brut_median,Montant brut_min,Montant brut_max,NET_mean,NET_median,NET_min,NET_max
0,A3,5.652122e+05,559159.014,312833.752,910643.680,4.915643e+05,477214.245648,283726.382896,7.773214e+05
1,A4,7.388218e+05,731640.388,340284.588,1226606.168,6.336566e+05,626960.654496,306651.569024,9.930360e+05
2,A5,7.548189e+05,717311.574,379077.796,1284328.164,6.445360e+05,618301.369936,339646.663808,1.032724e+06
3,A6,8.515740e+05,789874.166,473043.836,1582666.984,7.160144e+05,674317.627984,406827.976928,1.253473e+06
4,A7,1.377912e+06,1013602.908,646209.824,3087477.680,1.145789e+06,829861.614604,531085.866752,2.546068e+06
5,B1,3.180962e+05,316414.164,241688.036,428122.860,2.928641e+05,289698.420832,226743.466888,3.920049e+05
6,B3,4.622090e+05,457372.650,272562.592,699928.528,4.123359e+05,406713.962640,253866.193216,6.105856e+05
7,C1,3.083891e+05,319742.268,213498.224,391823.616,2.933992e+05,304004.181944,201330.964192,3.748953e+05
8,C2,3.094970e+05,301336.496,217525.340,406589.708,2.938189e+05,285995.168416,204961.321720,3.868522e+05
9,C3,3.342051e+05,337765.962,220210.084,436121.892,3.120441e+05,308149.935492,207381.560072,4.087497e+05


In [33]:
avg_by_grade.to_excel(r'C:\Users\a_coulibaly\OneDrive - Metrics & Decisions\INS\CAE\Banque de données travailleurs\Data\Solde\Output\stat_amounts.xlsx', index=False, sheet_name='avg_by_grade')

In [34]:
with pd.ExcelWriter(r'C:\Users\a_coulibaly\OneDrive - Metrics & Decisions\INS\CAE\Banque de données travailleurs\Data\Solde\Output\stat_amounts.xlsx', engine='openpyxl', mode='a') as writer:
    avg_by_secteur_grade.to_excel(writer, sheet_name='avg_by_secteur_grade', index=False)